[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/10_troubleshooting.ipynb)

# 教程 10：故障排查与常见错误

一份生存指南，应对每个 MuJoCo 新手都会撞上的那些错误。

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import mujoco
import numpy as np
import mediapy as media

---
## 问题 1：渲染时黑屏

**症状：** 你渲染出一张图，但它全是黑的（或局部是黑的）。

**原因：** 你在手动设置 `qpos` 之后没有调用 `mj_forward()`，所以派生量（比如几何体（geom）的笛卡尔坐标）还没有被计算出来。

In [ ]:
xml = """
<mujoco>
  <worldbody>
    <light pos="0 0 1.5"/>
    <geom type="box" size=".2 .2 .2" pos="0 0 .2" rgba="1 0 0 1"/>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model)

# ❌ WRONG: rendering without mj_forward
renderer.update_scene(data)
bad_image = renderer.render().copy()

# ✅ CORRECT: call mj_forward first
mujoco.mj_forward(model, data)
renderer.update_scene(data)
good_image = renderer.render().copy()

renderer.close()
media.show_images([bad_image, good_image], titles=['❌ Without mj_forward', '✅ With mj_forward'])

**修复：** 在手动设置 `data.qpos` 或 `data.qvel` 之后，务必调用 `mujoco.mj_forward(model, data)`。如果你在循环里调用 `mj_step()`，这一步会自动完成。

---
## 问题 2：仿真“爆炸”（NaN 或巨大数值）

**症状：** 位置变成 `inf` 或 `nan`，物体飞到无穷远。

**常见原因：**
1. 对这个模型来说仿真步长（timestep）太大
2. 控制力极大
3. 初始化时几何体相互穿透

In [ ]:
# Example: timestep too large
xml_stiff = """
<mujoco>
  <option timestep="0.1" gravity="0 0 -9.81"/>  <!-- 0.1s is way too large! -->
  <worldbody>
    <geom type="plane" size="5 5 0.1"/>
    <body pos="0 0 0.5">
      <joint type="free"/>
      <geom type="sphere" size="0.1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(xml_stiff)
d = mujoco.MjData(m)

print("Simulating with dt=0.1 (dangerously large):")
for i in range(20):
    mujoco.mj_step(m, d)
    z = d.qpos[2]
    if abs(z) > 100 or np.isnan(z):
        print(f"  Step {i}: z = {z} ← EXPLODED!")
        break
    elif i < 5 or i % 5 == 0:
        print(f"  Step {i}: z = {z:.4f}")

print("\n✅ Fix: Use timestep ≤ 0.005 (default is 0.002)")

In [ ]:
# How to detect instability in your code:
def check_simulation_health(data):
    """Call this periodically to catch explosions early."""
    if np.any(np.isnan(data.qpos)) or np.any(np.isnan(data.qvel)):
        return "NaN detected! Simulation diverged."
    if np.any(np.abs(data.qpos) > 1e6):
        return "Extreme positions! Likely unstable."
    if np.any(np.abs(data.qvel) > 1e6):
        return "Extreme velocities! Likely unstable."
    return "OK"

print("Use this helper to catch problems early!")
print(f"Health check: {check_simulation_health(d)}")

---
## 问题 3：物体互相穿透

**症状：** 本该碰撞的刚体（body）没有碰撞。

**常见原因：**
1. `contype`/`conaffinity` 不匹配（碰撞过滤）
2. 同一个刚体（body）内的几何体默认不会互相碰撞
3. 物体初始位置就严重穿透

In [ ]:
# Collision happens when: (geomA.contype & geomB.conaffinity) OR (geomB.contype & geomA.conaffinity)
# Default: contype=1, conaffinity=1 → everything collides with everything

# If you accidentally set contype=0:
xml_no_collide = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="2 2 0.1"/>  <!-- contype=1 by default -->
    <body pos="0 0 1">
      <joint type="free"/>
      <geom type="sphere" size="0.1" contype="0" conaffinity="0"/>  <!-- ❌ won't collide! -->
    </body>
  </worldbody>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(xml_no_collide)
d = mujoco.MjData(m)

# The ball falls through the floor
for _ in range(500):
    mujoco.mj_step(m, d)
print(f"Ball fell to z={d.qpos[2]:.2f} (below the ground!)")
print("\n✅ Fix: Ensure colliding geoms have matching contype & conaffinity")
print("   Default (contype=1, conaffinity=1) works for most cases.")

---
## 问题 4：机器人不动（控制没有效果）

**症状：** 你设置了 `data.ctrl`，但什么也没发生。

**常见原因：**
1. 忘了在 XML 里添加 `<actuator>`
2. 控制范围太小
3. 关节（joint）阻尼太大，把运动全吞掉了
4. 在控制循环之外设置 ctrl（只设置了一次）

In [ ]:
# Debugging checklist:
xml_debug = """
<mujoco>
  <worldbody>
    <body pos="0 0 1">
      <joint name="j1" type="hinge" axis="0 1 0" damping="0.1"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.03" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <motor name="m1" joint="j1" ctrlrange="-5 5" gear="1"/>
  </actuator>
</mujoco>
"""
m = mujoco.MjModel.from_xml_string(xml_debug)
d = mujoco.MjData(m)

print("=== Control Debugging Checklist ===")
print(f"  Number of actuators (model.nu): {m.nu}")
print(f"  Control range: {m.actuator_ctrlrange}")
print(f"  Actuator gear: {m.actuator_gear[:, 0]}")
print(f"  Joint damping: {m.jnt_stiffness}")
print(f"  data.ctrl shape: {d.ctrl.shape}")
print()
print("If nu=0, you forgot to add <actuator> in your XML!")
print("If ctrl has no effect, check gear and ctrlrange.")

---
## 问题 5：模型加载不了（XML 错误）

**常见 XML 错误：**

In [ ]:
# Example: common XML mistakes and how to fix them

bad_examples = {
    "Missing closing tag": '<mujoco><worldbody><geom type="sphere"/></mujoco>',
    "Actuator references nonexistent joint": '''
        <mujoco>
          <worldbody><body><joint name="j1"/><geom size="0.1"/></body></worldbody>
          <actuator><motor joint="wrong_name"/></actuator>
        </mujoco>''',
    "Body outside worldbody": '<mujoco><body><geom size="0.1"/></body></mujoco>',
}

for name, xml in bad_examples.items():
    try:
        mujoco.MjModel.from_xml_string(xml)
        print(f"  {name}: no error (unexpected)")
    except Exception as e:
        # Get first line of error
        error_msg = str(e).split('\n')[0][:80]
        print(f"❌ {name}")
        print(f"   Error: {error_msg}")
        print()

**XML 调试技巧：**
1. 从一个能跑通的最小模型开始，一次只添加一样东西
2. MuJoCo 的报错信息会告诉你是第几行——读它们！
3. 使用 XML 校验器，或带 XML 高亮的 IDE
4. 常见笔误：`fromto` 需要 6 个数字（x1 y1 z1 x2 y2 z2）

---
## 问题 6：本地渲染正常，服务器/Colab 上不行

**症状：** `mujoco.viewer.launch()` 失败，报显示相关的错误。

**原因：** 交互式查看器（viewer）需要一个显示屏（GUI）。服务器/Colab 上没有。

**修复：** 改用 `mujoco.Renderer`（离屏渲染）：

In [ ]:
import os

# Check if we have a display
has_display = 'DISPLAY' in os.environ or os.environ.get('MUJOCO_GL') == 'egl'

print("Rendering options:")
print(f"  Display available: {has_display}")
print()
print("  Local machine (has display):")
print("    → mujoco.viewer.launch(model, data)      # interactive 3D viewer")
print("    → mujoco.viewer.launch_passive(model, data)  # non-blocking")
print()
print("  Server/Colab (no display):")
print("    → renderer = mujoco.Renderer(model)      # offscreen")
print("    → renderer.render()                       # returns numpy array")
print("    → media.show_image(pixels)                # display in notebook")
print()
print("  Tip: Set MUJOCO_GL=egl for headless GPU rendering:")
print("    os.environ['MUJOCO_GL'] = 'egl'  # before importing mujoco")

---
## 问题 7：性能太慢

**提速技巧：**

In [ ]:
import time

xml = """
<mujoco>
  <option timestep="0.002"/>
  <worldbody>
    <geom type="plane" size="5 5 0.1"/>
    <body pos="0 0 1"><joint type="free"/><geom type="sphere" size="0.1"/></body>
  </worldbody>
</mujoco>
"""
m = mujoco.MjModel.from_xml_string(xml)
d = mujoco.MjData(m)

# Benchmark
n_steps = 100_000
t0 = time.time()
for _ in range(n_steps):
    mujoco.mj_step(m, d)
elapsed = time.time() - t0

sim_time = n_steps * m.opt.timestep
print(f"{n_steps:,} steps in {elapsed:.2f}s")
print(f"Simulated {sim_time:.0f}s of physics in {elapsed:.2f}s wall time")
print(f"Speed: {sim_time/elapsed:.0f}× real-time")
print()
print("Performance tips:")
print("  1. Don't render every step — render at 30-60 fps only")
print("  2. Use larger timestep if stable (0.005 instead of 0.002)")
print("  3. Reduce model complexity (fewer geoms/contacts)")
print("  4. For RL: use MJX (GPU) to batch thousands of envs")
print("  5. Avoid Python loops around mj_step — use native callbacks")

---
## 快速参考：常用操作

```python
# ─── Setup ───
model = mujoco.MjModel.from_xml_string(xml)   # or from_xml_path("file.xml")
data = mujoco.MjData(model)

# ─── Reset ───
mujoco.mj_resetData(model, data)               # reset to initial state
data.qpos[0] = 1.5                             # then set custom state
mujoco.mj_forward(model, data)                 # propagate (IMPORTANT!)

# ─── Simulate ───
data.ctrl[0] = 1.0                             # set control
mujoco.mj_step(model, data)                    # advance one timestep

# ─── Read state ───
pos = data.qpos[:]                             # all positions
vel = data.qvel[:]                             # all velocities
t = data.time                                  # current sim time

# ─── Named access ───
model.joint('shoulder').id                     # joint ID by name
data.body('robot').xpos                        # body position in world
data.site('end_effector').xpos                 # site position

# ─── Render ───
renderer = mujoco.Renderer(model, height=480, width=640)
mujoco.mj_forward(model, data)                 # needed before first render
renderer.update_scene(data)
pixels = renderer.render()                     # numpy array (H, W, 3)
renderer.close()                               # free resources

# ─── Contacts ───
n = data.ncon                                  # number of active contacts
force = np.zeros(6)
mujoco.mj_contactForce(model, data, 0, force)  # force at contact 0
```

---
## 还是卡住了？

1. **官方文档：** [mujoco.readthedocs.io](https://mujoco.readthedocs.io/)
2. **GitHub issues：** [github.com/google-deepmind/mujoco/issues](https://github.com/google-deepmind/mujoco/issues)
3. **XML 参考：** [mujoco.readthedocs.io/en/stable/XMLreference.html](https://mujoco.readthedocs.io/en/stable/XMLreference.html)
4. **Python 绑定：** [mujoco.readthedocs.io/en/stable/python.html](https://mujoco.readthedocs.io/en/stable/python.html)